# Editing a model in a notebook

A model displayed in a JupyterLab, Notebook 7, VS Code, or Colab cell is the
Simlin diagram editor, live -- with `pip install "pysimlin[notebook]"` (the
`notebook` extra is the widget host, anywidget; a bare `pysimlin` shows the
static diagram instead). This notebook opens a model file with
`simlin.open`, shows the editor, runs the model, edits it from Python, and
watches a change made by another tool arrive -- the way you would work on a
model together with Claude Code.

The one idea to keep in mind: **the file on disk is the shared truth.** Every
edit made in the editor is written to the file before the next cell runs;
every edit made to the file -- by a cell, by an agent, by `git checkout` --
shows up in the editor.

## Open a model file

Any XMILE (`.stmx`, `.xmile`), Vensim (`.mdl`), or Simlin JSON (`.sd.json`)
file works. Here we build the logistic-growth model from the README, save it
into a temporary directory, and open that file.

In [ ]:
import tempfile
from dataclasses import replace
from pathlib import Path

import simlin
from simlin import Aux, Flow, Stock

workdir = Path(tempfile.mkdtemp())
model_path = workdir / "logistic-growth.stmx"

project = simlin.Project.new(
    name="logistic-growth", sim_start=0, sim_stop=100, dt=0.25, time_units="years"
)
with project.get_model().edit() as (_, patch):
    patch.upsert(Stock(name="population", initial_equation="50", inflows=["net_growth"]))
    patch.upsert(Flow(name="net_growth", equation="population * fractional_growth"))
    patch.upsert(
        Aux(
            name="fractional_growth",
            equation="max_growth_rate * (1 - population / carrying_capacity)",
        )
    )
    patch.upsert(Aux(name="max_growth_rate", equation="0.08"))
    patch.upsert(Aux(name="carrying_capacity", equation="10000"))
project.save_as(model_path)

m = simlin.open(model_path)
print(m.path.name, "revision", m.revision)

## Display the editor

A model as the last expression of a cell shows the editor (`m.widget(height=..., theme=...)`
gives you a handle and options). Try it: drag a variable, add one from the
tool dial in the corner, click a variable to change its equation. Each edit is
written to the file as you make it (the equation editor's Save applies that one
equation); there is no project-level save step.

In [ ]:
m

## Run it

`m.run()` simulates the model as it is on disk right now, including anything
you just did in the editor.

In [ ]:
run = m.run()
print(f"final population: {run.results['population'].iloc[-1]:.0f}")
print("revision", m.revision, "dirty", m.dirty)

## Edit from Python

`m.edit()` applies a change through the same path: it is written to the file,
`revision` advances, and the editor above shows it with a short
"Updated from Python" notice.

In [ ]:
with m.edit() as (current, patch):
    patch.upsert(replace(current["carrying_capacity"], equation="12000"))

run = m.run()
print(f"final population: {run.results['population'].iloc[-1]:.0f}")
print("revision", m.revision)

## Collaborating with Claude Code

Point Claude Code (or any other tool: the `simlin` MCP server, a script,
`git checkout`) at `model_path` and let it edit the file. The open model polls
its file every half second: an external change is loaded in place, the editor
shows "Updated on disk", `revision` advances, and the next `m.run()` reflects
it. Your own saves are recognised by content and never round-trip as
external changes.

The cell below stands in for such a collaborator: a second, independent
handle on the same file changes an equation. `m.reload()` reads it right
away so this cell is deterministic; in a live session you would just watch
the editor update on its own.

In [ ]:
collaborator = simlin.open(model_path, watch=False)
with collaborator.edit() as (current, patch):
    patch.upsert(replace(current["max_growth_rate"], equation="0.12"))

m.reload()
print("revision", m.revision, "max_growth_rate =", m.get_variable("max_growth_rate").equation)

Selection flows the other way. Click a variable in the editor above, then
run the next cell: `m.selection` is the tuple of selected variable names, so
an agent driving cells can ask what you are looking at before it acts.

In [ ]:
print(m.selection)

Subscribe to `m.project.on_change` to react to changes from any source
(`edit`, `widget`, `disk`, `reload`) as they happen -- for example to re-run
the model and update a chart whenever the file changes.